# MosAIc: comparing baselines

The baseline is MosAIc's starting forecast, made from prices alone before any filings or news are read: the probability that the stock falls a lot, stays roughly flat or rises a lot over the next 21 trading days. Every ledger line is a shift away from it, so if the baseline is off, the whole forecast is off.

**Why this notebook.** In the Tree of Thought validation run, two days after each results announcement, the baseline scored worse than a fixed 31% / 38% / 31% guess. The results-day jump had just happened, the baseline counted it as "the stock is very jumpy right now" and predicted big moves, but the month after results is usually calm (11 of 21 months were flat).

**The versions compared.** Each adds one idea to the one before, so each step's effect can be seen on its own:

| Version | What it does |
| --- | --- |
| A | current baseline: recent daily moves, the last few days counting most, plus the typical results-day jump if results are expected within the month |
| B | A, but results-day moves are left out of "recent moves"; results count only through the results-day jump term |
| C | B mixed half and half with the past year's moves (without results days), so a few unusual weeks matter less |
| D | C adjusted by how calm or jumpy the month usually is at this point after results, learned from past years of all 30 stocks |
| Fixed guess | 30.85% / 38.3% / 30.85% every time, the probabilities if the coming month moves like an average month of the past year |

**Why this can be tested on thousands of forecasts.** The baseline needs no language model, only arithmetic on prices. So all 30 stocks are tested every week from 2016, about 16,000 forecasts, against what actually happened. Version D only learns from windows that ended before each forecast date, so every forecast is a fair out-of-sample test.

**Later:** once the FirstRate option data is loaded, the same forecasts can also be scored against the option-implied probabilities, the main metric of the design.

In [ ]:
!pip install -q yfinance scipy

## 1. Settings

In [ ]:
USER_AGENT = "Your Name your.email@example.com"     # required by the SEC

TICKERS = ["AAPL", "MSFT", "NVDA", "INTC", "GOOGL", "META", "NFLX", "AMZN", "TSLA", "NKE",
           "JPM", "GS", "V", "JNJ", "UNH", "LLY", "PG", "KO", "WMT", "XOM", "CVX",
           "CAT", "BA", "LMT", "NEE", "DUK", "PLD", "AMT", "LIN", "FCX"]

PRICE_START = "2012-01-01"          # price history to download
TEST_START, TEST_END = "2016-01-04", "2026-09-25"   # weekly forecast dates (Mondays)

# Forecast definition (Checkpoint 1.1)
HORIZON = 21
SIGMA_LOOKBACK = 252
BUCKET_K = 0.5

# Baseline pieces
EWMA_DECAY = 0.94                   # recent days weigh more (version A, as in the notebooks so far)
BLEND_RECENT = 0.5                  # version C: weight on recent moves; the rest on the past year
EARNINGS_GAP_DAYS = 91              # next results expected this long after the last
EARNINGS_WINDOW_DAYS = 30           # "within the month" for the results-day jump term
AFTER_CLOSE_UTC_HOUR = 20           # results filed at or after this hour (UTC) move the stock the next day

# Version D: calm-down profile, by trading days since the last results
CALM_BINS = [(1, 5), (6, 15), (16, 30), (31, 45), (46, 63), (64, 999)]
CALM_MIN_WINDOWS = 50               # fewer past windows than this: no adjustment

FIXED_GUESS = {"fall": 30.85, "flat": 38.3, "gain": 30.85}
N_BOOT = 1000

In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from scipy.stats import norm

OUTCOMES = ("fall", "flat", "gain")
VERSIONS = ("A", "B", "C", "D", "Fixed guess")

## 2. Data: results dates and prices

Results dates come from the SEC's list of each company's filings: an 8-K with item 2.02 is a results announcement. Only the list is downloaded, not the documents, so this takes a minute. Prices come from Yahoo Finance (adjusted closes).

In [ ]:
import json, re, time, requests

HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}

def sec_get(url):
    time.sleep(0.15)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r

def results_times(cik):
    """Acceptance times (UTC) of every results 8-K (item 2.02), over the full filing history."""
    sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()
    blocks = [sub["filings"]["recent"]] + [sec_get(f"https://data.sec.gov/submissions/{p['name']}").json()
                                           for p in sub["filings"].get("files", [])]
    out = []
    for b in blocks:
        for form, items, acc in zip(b["form"], b.get("items", [""] * len(b["form"])), b["acceptanceDateTime"]):
            if form == "8-K" and "2.02" in str(items):
                out.append(datetime.fromisoformat(acc.replace("Z", "+00:00")).astimezone(timezone.utc))
    return sorted(out)

cik_map = {v["ticker"].upper(): int(v["cik_str"])
           for v in sec_get("https://www.sec.gov/files/company_tickers.json").json().values()}
RESULTS = {t: results_times(cik_map[t]) for t in TICKERS}
for t in TICKERS:
    print(f"{t}: {len(RESULTS[t])} results filings since {RESULTS[t][0].date() if RESULTS[t] else '-'}")

In [ ]:
import yfinance as yf
PRICES = {}
for t in TICKERS:
    px = yf.download(t, start=PRICE_START, progress=False, auto_adjust=True)["Close"]
    PRICES[t] = px.squeeze().dropna()
print({t: len(p) for t, p in PRICES.items()})

## 3. One stock's price history

For each stock, the notebook keeps its daily closes, its daily log returns and which days were **results days**. A results filing made after the market closes moves the stock the next trading day; one made during the day moves it the same day.

Conventions, the same as in the other notebooks:

- A forecast "as of" a date is made at the start of that day, so only closes on earlier days are known.
- The thresholds for "large" moves use the past year of all daily returns, results days included, as in the forecast definition. Only the baselines' estimate of the coming month's movement changes between versions, never the thresholds.

In [ ]:
class Stock:
    def __init__(self, ticker, px, results):
        px = px.dropna()
        self.ticker = ticker
        self.dates = np.array(px.index.values, dtype="datetime64[D]")
        self.logp = np.log(px.to_numpy(dtype=float))
        self.ret = np.concatenate([[np.nan], np.diff(self.logp)])      # ret[t] = move from close t-1 to close t
        self.results = sorted(results)
        self.reaction = np.zeros(len(self.dates), dtype=bool)
        self.reaction_pos = []
        for a in self.results:
            d = np.datetime64(a.date(), "D")
            side = "right" if a.hour >= AFTER_CLOSE_UTC_HOUR else "left"
            t = int(np.searchsorted(self.dates, d, side=side))
            if 1 <= t < len(self.dates):
                self.reaction[t] = True
                self.reaction_pos.append(t)
        self.reaction_pos = sorted(set(self.reaction_pos))
        ok = ~self.reaction & ~np.isnan(self.ret)
        self.cum_sq = np.concatenate([[0.0], np.cumsum(np.where(ok, self.ret, 0.0) ** 2)])
        self.cum_n = np.concatenate([[0], np.cumsum(ok)])

    def position(self, as_of):
        """Index of the first trading day on or after as_of: closes before it are known."""
        return int(np.searchsorted(self.dates, np.datetime64(as_of.date(), "D"), side="left"))

    def can_forecast(self, i):
        return i - 1 >= SIGMA_LOOKBACK and i + HORIZON - 1 < len(self.dates)

    def sigma_year(self, i):
        return float(np.std(self.ret[i - SIGMA_LOOKBACK:i]) * math.sqrt(HORIZON))

    def thresholds(self, i):
        s = self.sigma_year(i)
        return -BUCKET_K * s, BUCKET_K * s

    def realised_bucket(self, i):
        lo, hi = self.thresholds(i)
        r = self.logp[i + HORIZON - 1] - self.logp[i - 1]
        return "fall" if r < lo else "gain" if r > hi else "flat"

    def year_ex_var(self, i):
        """Average squared daily move over the past year, results days left out."""
        a = max(i - SIGMA_LOOKBACK, 0)
        n = self.cum_n[i] - self.cum_n[a]
        return float((self.cum_sq[i] - self.cum_sq[a]) / n) if n else float("nan")

    def ewma_var(self, i, leave_out_results):
        r = self.ret[i - SIGMA_LOOKBACK:i]
        w = EWMA_DECAY ** np.arange(len(r))[::-1]
        keep = ~np.isnan(r)
        if leave_out_results:
            keep &= ~self.reaction[i - SIGMA_LOOKBACK:i]
        return float(np.sum(w[keep] * r[keep] ** 2) / np.sum(w[keep]))

    def last_results(self, as_of):
        past = [a for a in self.results if a < as_of]
        return past[-1] if past else None

    def earnings_in_window(self, as_of):
        last = self.last_results(as_of)
        return bool(last) and as_of < last + timedelta(days=EARNINGS_GAP_DAYS) <= as_of + timedelta(days=EARNINGS_WINDOW_DAYS)

    def days_since_results(self, i):
        past = [t for t in self.reaction_pos if t <= i - 1]
        return i - past[-1] if past else None

    def typical_move(self, i):
        """Average size of the last 4 results-day moves known before position i."""
        past = [t for t in self.reaction_pos if t <= i - 1][-4:]
        return float(np.mean(np.abs(self.ret[past]))) if past else None

    def typical_move_a(self, as_of):
        """Version A's results-day move, as in the earlier notebooks: close on the filing date to the next close."""
        moves = []
        for a in [x for x in self.results if x < as_of][-4:]:
            d = np.datetime64(a.date(), "D")
            j = int(np.searchsorted(self.dates, d, side="right"))   # first close after the filing date
            if 1 <= j < self.position(as_of):
                moves.append(abs(self.logp[j] - self.logp[j - 1]))
        return float(np.mean(moves)) if moves else None

## 4. Versions A, B and C

**How long version A stays fooled.** Recent days count most, but not only the last few: with the current weighting, a day 40 trading days ago still carries about 8% of the weight of yesterday. A 12% results-day jump therefore still inflates A's estimate weeks later, for most of the quarter. B removes this by leaving results days out.

Each version gives a variance for the coming month (how much the stock is expected to move), turned into the three probabilities with the same thresholds. When results are expected within the month, each adds the typical results-day jump.

In [ ]:
def probs_from_var(var_month, lower, upper):
    s = math.sqrt(var_month)
    fall, gain = norm.cdf(lower / s), 1 - norm.cdf(upper / s)
    return {"fall": 100 * fall, "flat": 100 * (1 - fall - gain), "gain": 100 * gain}

def core_vars(stock, i):
    """Monthly variance before any results-day jump, for versions A, B and C."""
    return {"A": stock.ewma_var(i, leave_out_results=False) * HORIZON,
            "B": stock.ewma_var(i, leave_out_results=True) * HORIZON,
            "C": (BLEND_RECENT * stock.ewma_var(i, leave_out_results=True)
                  + (1 - BLEND_RECENT) * stock.year_ex_var(i)) * HORIZON}

def baselines(stock, as_of, calm=None):
    """Fall / flat / gain (percent) for every version at one forecast date."""
    i = stock.position(as_of)
    lower, upper = stock.thresholds(i)
    v = core_vars(stock, i)
    if calm is not None:
        k = stock.days_since_results(i)
        v["D"] = v["C"] * calm.factor(k, as_of)
    jump = stock.earnings_in_window(as_of)
    out = {}
    for name, var in v.items():
        move = (stock.typical_move_a(as_of) if name == "A" else stock.typical_move(i)) if jump else None
        out[name] = probs_from_var(var + (move or 0.0) ** 2, lower, upper)
    out["Fixed guess"] = dict(FIXED_GUESS)
    return out